# SAE seed similarity evaluation

This notebook is the interactive front end to the same tested modules used by the CLI. Run expensive stages once, then freely reload tables and make additional plots without re-running the base model.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sae_seed_similarity.config import load_config
from sae_seed_similarity.collect_activations import collect
from sae_seed_similarity.match_features import run as match_features
from sae_seed_similarity.compare_representations import run as compare_representations
from sae_seed_similarity.make_report import run as make_report
from sae_seed_similarity.metrics import activation_overlap, linear_cka
from sae_seed_similarity.storage import ArtifactStore
sns.set_theme(style='whitegrid')

In [ ]:
CONFIG_PATH = Path('../configs/pythia_160m_two_seed.yaml').resolve()
cfg = load_config(CONFIG_PATH)
store = ArtifactStore(cfg.output_path).ensure()
cfg.to_dict()

## Resumable pipeline stages

Each call reuses complete cached outputs unless `force: true` is set in YAML. Activation collection requires the model/checkpoints; matching and representation comparison reuse its cached activations.

In [ ]:
RUN_EXPENSIVE_STAGES = False
if RUN_EXPENSIVE_STAGES:
    collect(cfg)
    matches, overlaps = match_features(cfg)
    representation_summary = compare_representations(cfg)
    report_path = make_report(cfg)

## Load cached results

In [ ]:
matches = pd.read_parquet(store.root / 'hungarian_matches.parquet')
overlaps = pd.read_parquet(store.root / 'activation_overlap.parquet')
representation_summary = pd.read_csv(store.root / 'seed_pair_summary.csv')
display(representation_summary)
display(matches.head())
display(overlaps.head())

## Direct metric use

All metrics are standalone functions. Sparse latent caches are accepted directly, so exploratory analysis does not need dense 32,768-column arrays.

In [ ]:
X = store.load_latents(cfg.saes[0].name)
Y = store.load_latents(cfg.saes[1].name)
sample = np.arange(min(10_000, X.shape[0]))
print('raw CKA:', linear_cka(X[sample], Y[sample]))
pair = matches.iloc[0]
activation_overlap(X, Y, int(pair.feature_a), int(pair.feature_b))

## Example custom plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=overlaps, x='decoder_cosine', y='jaccard', alpha=.4, ax=axes[0])
axes[0].set_title('Matched feature geometry and firing overlap')
sns.histplot(data=overlaps, x='jaccard', bins=40, ax=axes[1])
axes[1].set_title('Activation Jaccard distribution')
plt.tight_layout()